
# Fairness

In [0]:
%pip install fairlearn

In [0]:
import pandas as pd
import numpy as np

from fairlearn.datasets import fetch_adult
from fairlearn.metrics import MetricFrame, demographic_parity_difference, selection_rate

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


from fairlearn.reductions import GridSearch, DemographicParity


In [0]:
data_dictionary = fetch_adult(as_frame=True)

In [0]:

SENSITIVE_VAR = "sex"
X = data_dictionary.data
S = X[SENSITIVE_VAR]
y = (data_dictionary.target == ">50K").astype(int)


X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(X, y, S, test_size=0.2, random_state=0)

In [0]:
X.columns

In [0]:

numerical_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]

categorical_features = ["workclass", "education", "marital-status", "relationship", "race", "native-country"]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [0]:

lr = LogisticRegression()
lr.fit(X_train_processed, y_train)

y_test_unmitigated = lr.predict(X_test_processed)

In [0]:
from sklearn.metrics import accuracy_score

metrics_fns = {
    "accuracy": accuracy_score,
    "selection_rate": selection_rate
}

unmitigated_metrics = MetricFrame(
    metrics=metrics_fns, 
    y_true=y_test, 
    y_pred=y_test_unmitigated,
    sensitive_features=S_test
)
unmitigated_metrics.overall

In [0]:
0.248728 * (1-0.33616542123042276) + 0.074604 * 0.33616542123042276

In [0]:
unmitigated_metrics.by_group

In [0]:
demographic_parity_difference(y_test, y_test_unmitigated, sensitive_features=S_test)

In [0]:
3284 / (6485 + 3284)

In [0]:
S_test.value_counts()


## Fair Model

In [0]:
constrains = DemographicParity()

gs = GridSearch(
    LogisticRegression(),
    constraints=constrains,
    constraint_weight=0.9,
)

X_train_dense = X_train_processed.toarray()
gs.fit(X_train_dense, y_train, sensitive_features=S_train)

In [0]:
y_pred_mitidated = gs.predict(X_test_processed.toarray())
mitigated_metrics = MetricFrame(
    metrics=metrics_fns, 
    y_true=y_test, 
    y_pred=y_pred_mitidated,
    sensitive_features=S_test
)

In [0]:
mitigated_metrics.by_group

In [0]:
mitigated_metrics.overall